# 03 — Modelado Predictivo

**Objetivo:** Entrenar, evaluar y guardar los modelos de clasificacion para:
- Prediccion de cesarea (CESAREA)
- Prediccion de bajo peso al nacer (BAJO_PESO)

Ejecute `01_limpieza_datos.ipynb` primero para generar el dataset procesado.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay,
    classification_report, roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import FEATURES, TARGET_CESAREA, TARGET_BAJO_PESO
from src.train import train_model, get_feature_importance

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'nac2018_cleaned.csv'
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
print(f'Dataset cargado: {df.shape[0]:,} filas')
print(f"Tasa CESAREA:    {df['CESAREA'].mean():.1%}")
print(f"Tasa BAJO_PESO:  {df['BAJO_PESO'].mean():.1%}")

## 1. Modelo — Cesarea

Se evaluan tres clasificadores con validacion cruzada estratificada (5 folds).
El mejor segun ROC-AUC se guarda en `models/cesarea_pipeline.pkl`.

In [ ]:
pipeline_ces, results_ces = train_model(
    df, TARGET_CESAREA, 'cesarea_pipeline.pkl', cv=5
)
print('\nComparacion de modelos (Cesarea):')
pd.DataFrame(results_ces).T.round(4)

## 2. Evaluacion del modelo de Cesarea

In [ ]:
X = df[FEATURES].dropna()
y_ces = df.loc[X.index, TARGET_CESAREA]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_ces, test_size=0.20, stratify=y_ces, random_state=42
)

y_pred = pipeline_ces.predict(X_test)
y_prob = pipeline_ces.predict_proba(X_test)[:, 1]

print(f'ROC-AUC (test): {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred,
      target_names=['Espontaneo', 'Cesarea']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=axes[0],
    name='Modelo Cesarea', color='#F44336'
)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Curva ROC — Cesarea', fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=axes[1],
    display_labels=['Espontaneo', 'Cesarea'],
    colorbar=False, cmap='Reds'
)
axes[1].set_title('Matriz de Confusion — Cesarea', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Importancia de variables — Cesarea

In [ ]:
fi_ces = get_feature_importance(pipeline_ces)
if fi_ces is not None:
    top = fi_ces.head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top['feature'][::-1], top['importance'][::-1],
            color='#F44336', alpha=0.85, edgecolor='white')
    ax.set_title('Top 15 Variables — Modelo Cesarea', fontweight='bold')
    ax.set_xlabel('Importancia')
    plt.tight_layout()
    plt.show()
    print(top.to_string(index=False))
else:
    print('Importancia no disponible para este tipo de modelo.')

## 4. Modelo — Bajo Peso al Nacer

Mismo proceso de seleccion. El modelo se guarda en `models/bajo_peso_pipeline.pkl`.

In [ ]:
pipeline_bp, results_bp = train_model(
    df, TARGET_BAJO_PESO, 'bajo_peso_pipeline.pkl', cv=5
)
print('\nComparacion de modelos (Bajo Peso):')
pd.DataFrame(results_bp).T.round(4)

## 5. Evaluacion del modelo de Bajo Peso

In [ ]:
y_bp = df.loc[X.index, TARGET_BAJO_PESO]
_, X_test2, _, y_test2 = train_test_split(
    X, y_bp, test_size=0.20, stratify=y_bp, random_state=42
)

y_pred2 = pipeline_bp.predict(X_test2)
y_prob2 = pipeline_bp.predict_proba(X_test2)[:, 1]

print(f'ROC-AUC (test): {roc_auc_score(y_test2, y_prob2):.4f}')
print()
print(classification_report(y_test2, y_pred2,
      target_names=['Normal', 'Bajo Peso']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

RocCurveDisplay.from_predictions(
    y_test2, y_prob2, ax=axes[0],
    name='Modelo Bajo Peso', color='#FF9800'
)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Curva ROC — Bajo Peso', fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test2, y_pred2, ax=axes[1],
    display_labels=['Normal', 'Bajo Peso'],
    colorbar=False, cmap='Oranges'
)
axes[1].set_title('Matriz de Confusion — Bajo Peso', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Importancia de variables — Bajo Peso

In [ ]:
fi_bp = get_feature_importance(pipeline_bp)
if fi_bp is not None:
    top = fi_bp.head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top['feature'][::-1], top['importance'][::-1],
            color='#FF9800', alpha=0.85, edgecolor='white')
    ax.set_title('Top 15 Variables — Modelo Bajo Peso', fontweight='bold')
    ax.set_xlabel('Importancia')
    plt.tight_layout()
    plt.show()
    print(top.to_string(index=False))
else:
    print('Importancia no disponible para este tipo de modelo.')

## 7. Resumen y siguientes pasos

In [ ]:
from pathlib import Path

models_dir = PROJECT_ROOT / 'models'
saved = list(models_dir.glob('*.pkl'))
print('Modelos guardados:')
for p in saved:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name}  ({size_mb:.1f} MB)')

print('\nPara iniciar la interfaz grafica:')
print('  uv run streamlit run app/main.py')